In [1]:
import os
import requests
import json
import pandas as pd
from dotenv import load_dotenv

# 1. Load environment variables
load_dotenv(dotenv_path="../.env")
USER_AGENT = os.getenv("SEC_USER_AGENT")

if not USER_AGENT or USER_AGENT == "Your Name (your.email@domain.com)":
    print("WARNING: Please update your .env file with a real Name and Email.")

headers = {"User-Agent": USER_AGENT}

# 2. Map the Ticker (WMT) to the SEC's CIK number
print("Fetching SEC Ticker to CIK mapping...")
tickers_url = "https://www.sec.gov/files/company_tickers.json"
response = requests.get(tickers_url, headers=headers)
ticker_data = response.json()

wmt_cik_str = ""
for key, company in ticker_data.items():
    if company["ticker"] == "WMT":
        # SEC requires CIKs to be padded with zeros to 10 digits in their URLs
        wmt_cik_str = str(company["cik_str"]).zfill(10) 
        break

print(f"Walmart (WMT) CIK mapped to: {wmt_cik_str}")

# 3. Fetch all financial facts for Walmart
print(f"Fetching Company Facts for CIK {wmt_cik_str} (This may take 5-10 seconds)...")
facts_url = f"https://data.sec.gov/api/xbrl/companyfacts/CIK{wmt_cik_str}.json"
facts_response = requests.get(facts_url, headers=headers)
wmt_facts = facts_response.json()

# 4. Save the raw JSON locally so we don't spam the SEC API every time we run the notebook
raw_data_path = "../data/raw/wmt_company_facts.json"
with open(raw_data_path, "w") as f:
    json.dump(wmt_facts, f)

# 5. Inspect the success
gaap_metrics = list(wmt_facts['facts']['us-gaap'].keys())
print(f"✅ Successfully saved data to {raw_data_path}")
print(f"📊 Total US-GAAP accounting concepts retrieved: {len(gaap_metrics)}")
print(f"🔍 Sample of available metrics: {gaap_metrics[:5]}")

Fetching SEC Ticker to CIK mapping...
Walmart (WMT) CIK mapped to: 0000104169
Fetching Company Facts for CIK 0000104169 (This may take 5-10 seconds)...
✅ Successfully saved data to ../data/raw/wmt_company_facts.json
📊 Total US-GAAP accounting concepts retrieved: 478
🔍 Sample of available metrics: ['AccountsPayableCurrent', 'AccountsReceivableNet', 'AccrualForTaxesOtherThanIncomeTaxesCurrent', 'AccrualForTaxesOtherThanIncomeTaxesCurrentAndNoncurrent', 'AccruedIncomeTaxesCurrent']


In [3]:
import pandas as pd

us_gaap_data = wmt_facts['facts']['us-gaap']

def extract_annual_metric(possible_tags, readable_name):
    """Tries a list of XBRL tags and extracts annual data from the first one that works."""
    for tag in possible_tags:
        if tag in us_gaap_data and 'USD' in us_gaap_data[tag]['units']:
            data = us_gaap_data[tag]['units']['USD']
            # Filter for annual reports (10-K) and full fiscal years (FY)
            annual_data = [d for d in data if d.get('form') == '10-K' and d.get('fp') == 'FY']
            df = pd.DataFrame(annual_data)
            
            if not df.empty:
                df['filed'] = pd.to_datetime(df['filed'])
                df = df.sort_values('filed', ascending=False).drop_duplicates(subset=['fy'])
                print(f"✅ Found '{readable_name}' using tag: {tag}")
                return df[['fy', 'val']].rename(columns={'val': readable_name, 'fy': 'FiscalYear'})
                
    print(f"⚠️ Could not find any valid tags for {readable_name}.")
    return pd.DataFrame()

# FP&A standard metrics mapped to common SEC XBRL variations
metrics_map = {
    "Revenue": ["Revenues", "SalesRevenueNet", "RevenueFromContractWithCustomerExcludingAssessedTax"],
    "COGS": ["CostOfRevenue", "CostOfGoodsAndServicesSold", "CostOfGoodsSold", "CostOfSales"],
    "OperatingIncome": ["OperatingIncomeLoss"],
    "NetIncome": ["NetIncomeLoss"]
}

# Build the master DataFrame
wmt_pl = pd.DataFrame(columns=['FiscalYear'])

for name, tags in metrics_map.items():
    df_metric = extract_annual_metric(tags, name)
    if not df_metric.empty:
        if wmt_pl.empty:
            wmt_pl = df_metric
        else:
            wmt_pl = pd.merge(wmt_pl, df_metric, on='FiscalYear', how='outer')

# Sort chronologically and drop very old data (keep last 10 years for forecasting)
wmt_pl = wmt_pl.sort_values('FiscalYear').reset_index(drop=True)
wmt_pl = wmt_pl[wmt_pl['FiscalYear'] >= 2015].reset_index(drop=True)

# Calculate Gross Profit (Revenue - COGS)
if 'Revenue' in wmt_pl.columns and 'COGS' in wmt_pl.columns:
    wmt_pl['GrossProfit'] = wmt_pl['Revenue'] - wmt_pl['COGS']

# Convert raw dollars to Billions ($B)
cols_to_convert = [col for col in wmt_pl.columns if col != 'FiscalYear']
wmt_pl[cols_to_convert] = wmt_pl[cols_to_convert] / 1e9

# Reorder columns for a standard P&L view
expected_cols = ['FiscalYear', 'Revenue', 'COGS', 'GrossProfit', 'OperatingIncome', 'NetIncome']
available_cols = [c for c in expected_cols if c in wmt_pl.columns]
wmt_pl = wmt_pl[available_cols]

print("\n📊 Walmart 10-Year P&L Extracted (in $ Billions):")
print(wmt_pl.tail(5).to_string(index=False))

# Save to interim data folder
wmt_pl.to_csv("../data/interim/wmt_pl_annual.csv", index=False)

✅ Found 'Revenue' using tag: Revenues
✅ Found 'COGS' using tag: CostOfRevenue
✅ Found 'OperatingIncome' using tag: OperatingIncomeLoss
✅ Found 'NetIncome' using tag: NetIncomeLoss

📊 Walmart 10-Year P&L Extracted (in $ Billions):
 FiscalYear  Revenue    COGS  GrossProfit  OperatingIncome  NetIncome
       2022  523.964 420.315      103.649           22.548     13.510
       2023  611.289 420.315      190.974           20.428     13.510
       2024  648.125 490.142      157.983           27.012     15.511
       2025  680.985 511.753      169.232           29.348     19.436
       2026  713.163 535.395      177.768           29.825     21.893
